# 030 - Gerar relat?rios para todas as UFs

Este notebook ? um orquestrador. Ele mant?m o notebook `020_gerar_relatorio_dados.ipynb` linear, executando-o uma vez para cada UF desejada por meio de vari?veis de ambiente.

## 1. Importar bibliotecas

In [1]:
from datetime import datetime
from pathlib import Path
import os
import subprocess
import sys

import pandas as pd

pd.set_option("display.max_columns", 120)
pd.set_option("display.width", 160)

## 2. Configurar par?metros de execu??o

- `UFS_RELATORIO = []` gera relat?rios para todas as UFs encontradas na base consolidada.
- Para gerar apenas algumas UFs, preencha a lista, por exemplo: `UFS_RELATORIO = ["MG", "SC"]`.
- Os mesmos par?metros tamb?m podem ser passados por vari?vel de ambiente, o que facilita testes automatizados.

In [2]:
# Lista vazia significa: usar todas as UFs dispon?veis em indicadores_brasil_uf.csv.
UFS_RELATORIO = []

# Opcional: fixe uma compet?ncia intermedi?ria, como "202605". Se ficar None, usa a pasta mais recente.
ANO_MES_INTERMEDIARIO = None

# Opcional: limite de UFs para testes r?pidos. Use None para n?o limitar.
LIMITE_UFS = None

# Quando True, interrompe o lote no primeiro erro. Quando False, registra o erro e segue para a pr?xima UF.
PARAR_EM_ERRO = False

RUN_TIMESTAMP = datetime.now()
RUN_YYYYMM = RUN_TIMESTAMP.strftime("%Y%m")
RUN_DATETIME = RUN_TIMESTAMP.strftime("%Y%m%d%H%M%S")

CSV_SEP = ";"
CSV_ENCODING = "utf-8-sig"

# Sobrescritas por ambiente. ?til para testes sem editar o notebook.
ufs_env = os.environ.get("UFS_RELATORIO", "").strip()
if ufs_env:
    UFS_RELATORIO = [uf.strip().upper() for uf in ufs_env.split(",") if uf.strip()]

ANO_MES_INTERMEDIARIO = os.environ.get("ANO_MES_INTERMEDIARIO") or ANO_MES_INTERMEDIARIO

limite_env = os.environ.get("LIMITE_UFS", "").strip()
if limite_env:
    LIMITE_UFS = int(limite_env)

parar_env = os.environ.get("PARAR_EM_ERRO", "").strip().lower()
if parar_env:
    PARAR_EM_ERRO = parar_env in ["1", "true", "sim", "s", "yes"]

print(f"Data/hora da execu??o: {RUN_TIMESTAMP:%d/%m/%Y %H:%M:%S}")
print(f"UFs informadas: {UFS_RELATORIO if UFS_RELATORIO else 'todas'}")
print(f"Compet?ncia intermedi?ria: {ANO_MES_INTERMEDIARIO or 'mais recente'}")
print(f"Limite de UFs: {LIMITE_UFS or 'sem limite'}")

Data/hora da execu??o: 05/05/2026 09:11:31
UFs informadas: ['MG']
Compet?ncia intermedi?ria: mais recente
Limite de UFs: sem limite


## 3. Resolver caminhos

O orquestrador localiza a raiz do projeto, a pasta de dados intermedi?rios e o notebook `020`, que ser? executado como template.

In [3]:
inicio = Path.cwd().resolve()
PROJECT_ROOT = inicio

for candidato in [inicio, *inicio.parents]:
    tem_notebooks = (candidato / "notebooks").exists()
    tem_config = (candidato / "config" / "ufs.csv").exists()
    tem_requirements = (candidato / "requirements.txt").exists()
    if tem_notebooks and (tem_config or tem_requirements):
        PROJECT_ROOT = candidato
        break

NOTEBOOK_RELATORIO = PROJECT_ROOT / "notebooks" / "020_gerar_relatorio_dados.ipynb"
if not NOTEBOOK_RELATORIO.exists():
    raise FileNotFoundError(f"Notebook 020 n?o encontrado: {NOTEBOOK_RELATORIO}")

INTERMEDIARIOS_ROOT = PROJECT_ROOT / "dados_intermediarios"
if ANO_MES_INTERMEDIARIO is None:
    candidatos = sorted([p for p in INTERMEDIARIOS_ROOT.iterdir() if p.is_dir()]) if INTERMEDIARIOS_ROOT.exists() else []
    if not candidatos:
        raise FileNotFoundError(f"Nenhuma pasta encontrada em {INTERMEDIARIOS_ROOT}. Execute primeiro o notebook 010.")
    INPUT_DIR = candidatos[-1]
else:
    INPUT_DIR = INTERMEDIARIOS_ROOT / ANO_MES_INTERMEDIARIO

CONSOLIDADO_DIR = INPUT_DIR / "consolidado"
INDICADORES_PATH = CONSOLIDADO_DIR / "indicadores_brasil_uf.csv"
if not INDICADORES_PATH.exists():
    raise FileNotFoundError(f"Base consolidada n?o encontrada: {INDICADORES_PATH}")

OUTPUT_DIR = PROJECT_ROOT / "relatorios_gerados" / INPUT_DIR.name
LOGS_DIR = OUTPUT_DIR / "logs"
EXECUTADOS_DIR = PROJECT_ROOT / "notebooks" / "executados" / INPUT_DIR.name

for pasta in [OUTPUT_DIR, LOGS_DIR, EXECUTADOS_DIR]:
    pasta.mkdir(parents=True, exist_ok=True)

print(f"Raiz do projeto: {PROJECT_ROOT}")
print(f"Notebook template: {NOTEBOOK_RELATORIO}")
print(f"Entrada intermedi?ria: {INPUT_DIR}")
print(f"Sa?da dos relat?rios: {OUTPUT_DIR}")
print(f"Notebooks executados: {EXECUTADOS_DIR}")

Raiz do projeto: C:\Users\marce\OneDrive - Ministério da Agricultura e Pecuária\LAB_DATA_MDA\20260424_relatorio_dados\relatorio_dados_damei
Notebook template: C:\Users\marce\OneDrive - Ministério da Agricultura e Pecuária\LAB_DATA_MDA\20260424_relatorio_dados\relatorio_dados_damei\notebooks\020_gerar_relatorio_dados.ipynb
Entrada intermedi?ria: C:\Users\marce\OneDrive - Ministério da Agricultura e Pecuária\LAB_DATA_MDA\20260424_relatorio_dados\relatorio_dados_damei\dados_intermediarios\202605
Sa?da dos relat?rios: C:\Users\marce\OneDrive - Ministério da Agricultura e Pecuária\LAB_DATA_MDA\20260424_relatorio_dados\relatorio_dados_damei\relatorios_gerados\202605
Notebooks executados: C:\Users\marce\OneDrive - Ministério da Agricultura e Pecuária\LAB_DATA_MDA\20260424_relatorio_dados\relatorio_dados_damei\notebooks\executados\202605


## 4. Definir UFs a processar

A lista ? obtida da base consolidada quando `UFS_RELATORIO` est? vazia. Valores inv?lidos, como vazio ou `BR`, s?o removidos.

In [4]:
df_indicadores = pd.read_csv(INDICADORES_PATH, sep=CSV_SEP, encoding=CSV_ENCODING)
df_indicadores["uf"] = df_indicadores["uf"].astype(str).str.strip().str.upper()

ufs_disponiveis = sorted(
    uf for uf in df_indicadores["uf"].dropna().unique()
    if uf and uf not in ["BR", "NAN", "NONE", "NULL"]
)

if UFS_RELATORIO:
    ufs_invalidas = sorted(set(UFS_RELATORIO) - set(ufs_disponiveis))
    if ufs_invalidas:
        raise ValueError(f"UFs n?o encontradas na base consolidada: {ufs_invalidas}")
    ufs_processar = list(UFS_RELATORIO)
else:
    ufs_processar = ufs_disponiveis

if LIMITE_UFS is not None:
    ufs_processar = ufs_processar[:LIMITE_UFS]

print(f"UFs dispon?veis: {len(ufs_disponiveis)}")
print(f"UFs a processar: {len(ufs_processar)}")
print(ufs_processar)

UFs dispon?veis: 27
UFs a processar: 1
['MG']


## 5. Executar o notebook 020 para cada UF

Cada execu??o recebe `UF_INTERESSE` por vari?vel de ambiente e salva uma c?pia audit?vel do notebook executado em `notebooks/executados/AAAAMM`. O notebook `020` original n?o ? sobrescrito.

In [5]:
resultados = []

for posicao, uf in enumerate(ufs_processar, start=1):
    inicio_uf = datetime.now()
    timestamp_uf = inicio_uf.strftime("%Y%m%d%H%M%S")
    output_stem = f"020_gerar_relatorio_dados_{uf}_{timestamp_uf}"

    env = os.environ.copy()
    env["UF_INTERESSE"] = uf
    if ANO_MES_INTERMEDIARIO is not None:
        env["ANO_MES_INTERMEDIARIO"] = str(ANO_MES_INTERMEDIARIO)
    else:
        env.pop("ANO_MES_INTERMEDIARIO", None)

    comando = [
        sys.executable,
        "-m",
        "jupyter",
        "nbconvert",
        "--to",
        "notebook",
        "--execute",
        str(NOTEBOOK_RELATORIO),
        "--output-dir",
        str(EXECUTADOS_DIR),
        "--output",
        output_stem,
        "--ExecutePreprocessor.timeout=1200",
        "--ExecutePreprocessor.kernel_name=python3",
    ]

    print(f"[{posicao}/{len(ufs_processar)}] Gerando relat?rio para {uf}...")
    processo = subprocess.run(comando, env=env, capture_output=True, text=True)
    fim_uf = datetime.now()

    docx_gerado = None
    candidatos_docx = sorted(OUTPUT_DIR.glob(f"relatorio_dados_{uf}_*.docx"), key=lambda p: p.stat().st_mtime, reverse=True)
    if candidatos_docx:
        docx_gerado = candidatos_docx[0]

    status = "ok" if processo.returncode == 0 else "erro"
    mensagem = ""
    if processo.returncode != 0:
        mensagem = (processo.stderr or processo.stdout)[-2000:]
    else:
        mensagem = "Relat?rio gerado com sucesso."

    resultados.append({
        "uf": uf,
        "status": status,
        "inicio": inicio_uf.strftime("%Y-%m-%d %H:%M:%S"),
        "fim": fim_uf.strftime("%Y-%m-%d %H:%M:%S"),
        "duracao_segundos": round((fim_uf - inicio_uf).total_seconds(), 1),
        "notebook_executado": str((EXECUTADOS_DIR / f"{output_stem}.ipynb").resolve()),
        "relatorio_word": str(docx_gerado.resolve()) if docx_gerado else "",
        "mensagem": mensagem,
    })

    if status == "erro":
        print(f"Erro ao gerar {uf}.")
        if PARAR_EM_ERRO:
            raise RuntimeError(mensagem)
    else:
        print(f"Conclu?do: {docx_gerado}")

print("Execu??o em lote finalizada.")

[1/1] Gerando relat?rio para MG...
Conclu?do: C:\Users\marce\OneDrive - Ministério da Agricultura e Pecuária\LAB_DATA_MDA\20260424_relatorio_dados\relatorio_dados_damei\relatorios_gerados\202605\relatorio_dados_MG_20260505091138.docx
Execu??o em lote finalizada.


## 6. Salvar resumo da execu??o

In [6]:
df_resultados = pd.DataFrame(resultados)
resumo_path = LOGS_DIR / f"execucao_lote_relatorios_{RUN_DATETIME}.csv"
df_resultados.to_csv(resumo_path, sep=CSV_SEP, encoding=CSV_ENCODING, index=False)

print(f"Resumo salvo em: {resumo_path}")
display(df_resultados)

Resumo salvo em: C:\Users\marce\OneDrive - Ministério da Agricultura e Pecuária\LAB_DATA_MDA\20260424_relatorio_dados\relatorio_dados_damei\relatorios_gerados\202605\logs\execucao_lote_relatorios_20260505091131.csv


,uf,status,inicio,fim,duracao_segundos,notebook_executado,relatorio_word,mensagem
0,MG,ok,2026-05-05 09:11:31,2026-05-05 09:11:51,19.5,C:\Users\marce\OneDrive - Ministério da Agricu...,C:\Users\marce\OneDrive - Ministério da Agricu...,Relat?rio gerado com sucesso.
